# 06 - Model and data checks (for the methods appendix)

1. Data validation: rating lists vs equating tables, admitted students found in rating lists,
   raw-score recovery accuracy.
2. True-score model: fit at the top, implied test-retest reliability per subject-year-version.
3. Retest simulation preserves the score distribution (same version, and with version switching +
   equating error).

In [1]:
from common import *
from load import load_equating, load_rating, load_admissions
from scores import recover_raw, check_recovery
from simulate import subject_model, retest_equated, equating_delta

eq = load_equating()

## 1. Data validation

In [2]:
rows, rec_rows = [], []
for y in [2024, 2025, 2026]:
    r = load_rating(y)
    tot = eq[eq.year == y].groupby("subject").n.sum()
    cnt = r.subject.value_counts()
    for s in ["MATH", "ENG", "GEO", "HIST"]:
        rows.append({"year": y, "subject": s, "rating_list": int(cnt.get(s, 0)), "equating_tables": int(tot.get(s, 0))})
    if y >= 2025:
        a = load_admissions(y)
        rec = recover_raw(r, eq)
        c = check_recovery(rec, a)
        c.insert(0, "year", y)
        c["admitted_found_in_rating_list"] = a.exam_id.isin(set(r.exam_id)).mean()
        rec_rows.append(c)
counts = pd.DataFrame(rows)
counts["diff"] = counts.rating_list - counts.equating_tables
counts.to_csv(OUT / "t06_counts_check.csv", index=False)
recovery = pd.concat(rec_rows)
recovery.to_csv(OUT / "t06_raw_recovery_check.csv", index=False)
counts

,year,subject,rating_list,equating_tables,diff
0,2024,MATH,15766,15765,1
1,2024,ENG,37904,37899,5
2,2024,GEO,39600,39595,5
3,2024,HIST,19508,19504,4
4,2025,MATH,12979,12978,1
5,2025,ENG,32536,32533,3
6,2025,GEO,34119,34114,5
7,2025,HIST,17704,17701,3
8,2026,MATH,14249,14249,0
9,2026,ENG,33643,33640,3


In [3]:
recovery.round(4)

,year,subject,n_admitted,raw_exact,raw_within1,cells_multi_scaled,cells,admitted_found_in_rating_list
0,2025,ENG,28962,0.8511,0.9376,191,272,1.0
1,2025,GEO,30068,0.8460,0.8661,170,244,1.0
2,2025,HIST,14418,0.9311,0.9314,62,165,1.0
3,2025,MATH,9865,0.9172,1.0000,28,100,1.0
0,2026,ENG,30002,0.8641,0.9482,191,269,1.0
1,2026,GEO,31105,0.9142,0.9430,166,243,1.0
2,2026,HIST,14366,0.9429,0.9768,62,170,1.0
3,2026,MATH,11388,0.9924,0.9924,0,103,1.0


## 2. True-score model: fit and implied reliability
Reliability = correlation between the observed raw score and a simulated parallel-test raw score
(same version, baseline error), i.e. the test-retest reliability implied by the binomial error model.

In [4]:
rng = np.random.default_rng(7)
rows = []
for (y, s), e in eq[eq.subject.isin(["MATH", "ENG", "GEO"]) & (eq.year >= 2024)].groupby(["year", "subject"]):
    S = subject_model(e)
    for v, m in S.items():
        c = m["counts"].astype(int)
        raw = np.repeat(np.arange(len(c)), c)
        new = retest_equated(raw, np.full(len(raw), v), S, 1.0, False, None, rng)
        rows.append({"year": y, "subject": s, "version": v, "N": int(c.sum()),
                     "implied_reliability": np.corrcoef(m["equated"][raw], new)[0, 1],
                     "top3_observed": int(c[-3:].sum())})
rel = pd.DataFrame(rows)
fit = pd.read_csv(OUT / "t04_npmle_fit_checks.csv")
rel = rel.merge(fit[["year", "subject", "version", "top3_fit"]], on=["year", "subject", "version"], how="left")
rel.to_csv(OUT / "t06_model_reliability.csv", index=False)
rel.groupby(["subject"]).implied_reliability.agg(["min", "max"]).round(3)

,min,max
subject,,
ENG,0.964,0.974
GEO,0.902,0.922
MATH,0.878,0.926


In [5]:
rel.round(3)

,year,subject,version,N,implied_reliability,top3_observed,top3_fit
0,2024,ENG,1,7639,0.965,772,769.214
1,2024,ENG,2,7534,0.969,716,714.614
2,2024,ENG,3,7583,0.969,808,804.288
3,2024,ENG,4,7527,0.969,743,730.819
4,2024,ENG,5,7616,0.969,716,716.030
5,2024,GEO,1,7951,0.922,98,97.230
6,2024,GEO,2,7830,0.914,106,105.863
7,2024,GEO,3,7987,0.905,71,74.537
8,2024,GEO,4,7861,0.914,88,84.389
9,2024,GEO,5,7966,0.908,90,91.972


## 3. Does the simulated retest preserve the population score distribution? (2026)
Compared: number at or above max - 2.5 and max - 5.5 (equated scale) and the mean equated score of the top 2%.

In [6]:
rows = []
for s in ["MATH", "ENG", "GEO"]:
    S = subject_model(eq[(eq.year == 2026) & (eq.subject == s)])
    raws, vers, e0 = [], [], []
    for v, m in S.items():
        if not isinstance(v, (int, np.integer)):
            continue
        c = m["counts"].astype(int)
        r_ = np.repeat(np.arange(len(c)), c)
        raws.append(r_), vers.append(np.full(len(r_), v)), e0.append(m["equated"][r_])
    raw, ver, e0 = map(np.concatenate, (raws, vers, e0))
    M = S[1]["n"]

    def q(e):
        return [(e >= M - 2.5).sum(), (e >= M - 5.5).sum(), np.sort(e)[-int(0.02 * len(e)):].mean()]

    rows.append({"subject": s, "case": "observed", **dict(zip(["n_ge_max-2.5", "n_ge_max-5.5", "mean_top2pct"], q(e0)))})
    for name, sw, ee in [("retest, same version", False, False), ("retest, new version + equating error", True, True)]:
        sims = [q(retest_equated(raw, ver, S, 1.0, sw, equating_delta(S, rng) if ee else None, rng)) for _ in range(20)]
        rows.append({"subject": s, "case": name, **dict(zip(["n_ge_max-2.5", "n_ge_max-5.5", "mean_top2pct"], np.mean(sims, 0)))})
pres = pd.DataFrame(rows)
pres.to_csv(OUT / "t06_distribution_preservation.csv", index=False)
pres.round(2)

,subject,case,n_ge_max-2.5,n_ge_max-5.5,mean_top2pct
0,MATH,observed,268.00,664.00,49.92
1,MATH,"retest, same version",271.40,667.65,49.93
2,MATH,"retest, new version + equating error",277.85,651.85,49.90
3,ENG,observed,4345.00,9977.00,70.00
4,ENG,"retest, same version",4325.15,9964.90,70.00
5,ENG,"retest, new version + equating error",4329.85,9970.55,70.00
6,GEO,observed,1053.00,2978.00,59.05
7,GEO,"retest, same version",1052.50,2948.60,59.06
8,GEO,"retest, new version + equating error",1031.80,2997.60,59.05
